## Install Dependencies

In [ ]:
# Install dependencies
%pip install -q amplpy matplotlib

# Google Colab & Kaggle integration
from amplpy import AMPL, ampl_notebook
ampl = ampl_notebook(
    modules=["gurobi", "xpress", "highs", "gokestrel"], # modules to install
    license_uuid="default") # license to use

### AMPL Formulation

In [20]:
%%ampl_eval
reset;

# Sets
set FACTORIES;        # production locations
set PRODUCTS;         # product types

# Parameters
param required {FACTORIES, PRODUCTS};   # demand per factory-product
param capacity {FACTORIES};             # max production per factory

# Decision variables
var produce {FACTORIES, PRODUCTS} >= 0; # production quantity

# ❌ BUGGY (do not use): incorrect double indexing over FACTORIES
# subject to Capacity {f in FACTORIES}:
#    sum {f in FACTORIES, p in PRODUCTS} produce[f,p] <= capacity[f];

# ✅ CORRECT: sum only over products per factory
subject to Capacity_Correct {f in FACTORIES}:
    sum {p in PRODUCTS} produce[f,p] <= capacity[f];

# Demand constraints
subject to Demand {f in FACTORIES, p in PRODUCTS}:  
    produce[f,p] >= required[f,p];

# Objective: total production
maximize TotalProduction:
    sum {f in FACTORIES, p in PRODUCTS} produce[f,p];

In [21]:
%%ampl_eval
data;

# Sets
set FACTORIES := A B;
set PRODUCTS := widgets gadgets;

# Required production matrix
param required: widgets gadgets :=
    A   100     80
    B   150     120;

# Factory capacities
param capacity :=
A 1000
B 2000;

option solver highs;
solve;

HiGHS 1.11.0HiGHS 1.11.0: optimal solution; objective 3000
0 simplex iterations
0 barrier iterations
